# Test the five-way club-type model

Run the cells from top to bottom, choose an image, then run the prediction cell. The model classifies the club head as Driver, Fairway Wood, Hybrid, Iron, or Wedge.

In [ ]:
from io import BytesIO
from pathlib import Path
import sys

import matplotlib.pyplot as plt
from IPython.display import display
from PIL import Image
import ipywidgets as widgets

PROJECT_ROOT = next(
    parent
    for parent in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (parent / 'requirements.txt').is_file()
)
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from swingsight.club_cnn import classify_image

MODEL_PATH = PROJECT_ROOT / 'models' / 'trained' / 'club_type_5way.pt'
print(f'Model: {MODEL_PATH}')
if not MODEL_PATH.exists():
    print('Model not found. Put club_type_5way.pt in models/trained/, then run this cell again.')

In [ ]:
uploader = widgets.FileUpload(accept='image/*', multiple=False)
display(widgets.HTML('<b>Choose a club image:</b>'))
display(uploader)

In [ ]:
if not MODEL_PATH.exists():
    raise FileNotFoundError(f'Model not found: {MODEL_PATH}')
if not uploader.value:
    raise ValueError('Choose an image above, then run this cell.')

uploaded_files = uploader.value
uploaded = (
    next(iter(uploaded_files.values()))
    if isinstance(uploaded_files, dict)
    else uploaded_files[0]
)
image = Image.open(BytesIO(uploaded['content'])).convert('RGB')
prediction = classify_image(
    image,
    checkpoint_path=MODEL_PATH,
    expected_task='club_type_5way',
)
if not prediction.available:
    raise RuntimeError(prediction.reasoning)

display_names = {
    'driver': 'Driver',
    'wood': 'Fairway Wood',
    'hybrid': 'Hybrid',
    'iron': 'Iron',
    'wedge': 'Wedge',
}
label = display_names.get(prediction.label, prediction.label.title())
print(f'Prediction: {label} ({prediction.confidence:.1%} confidence)')

fig, (image_ax, scores_ax) = plt.subplots(1, 2, figsize=(12, 5))
image_ax.imshow(image)
image_ax.set_title(label)
image_ax.axis('off')

scores = sorted(prediction.probabilities.items(), key=lambda item: item[1])
scores_ax.barh([display_names.get(name, name.title()) for name, _ in scores], [score for _, score in scores])
scores_ax.set_xlim(0, 1)
scores_ax.set_xlabel('Confidence')
scores_ax.set_title('All model scores')
plt.tight_layout()
plt.show()